# Training Dynamics for LLM Fine-Tuning

This notebook provides hands-on examples for Module 05, covering:

1. Supervised Fine-Tuning (SFT) workflow
2. Learning rate finder
3. LoRA configuration and training
4. Gradient accumulation and memory optimization
5. Training monitoring and logging
6. Catastrophic forgetting prevention
7. Multi-GPU setup with Accelerate

## Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers datasets accelerate peft wandb torch

import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 1. Supervised Fine-Tuning Workflow

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

# Use a small model for demonstration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Use float32 for CPU, bfloat16 for GPU
    device_map="auto" if torch.cuda.is_available() else None,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Load a small dataset for demonstration
dataset = load_dataset("mlabonne/FineTome-100k", split="train[:500]")

print(f"Dataset size: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample entry:")
print(dataset[0])

In [ ]:
# Tokenize dataset
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text", "question", "response"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Tokenized dataset: {len(tokenized_dataset)} samples")
print(f"Input shape: {tokenized_dataset[0]['input_ids'].shape}")

## 2. Learning Rate Finder

In [ ]:
from transformers import TrainerCallback

class LRFinderCallback(TrainerCallback):
    """Callback to find optimal learning rate."""
    
    def __init__(self, start_lr=1e-8, end_lr=1.0):
        self.lrs = []
        self.losses = []
        self.start_lr = start_lr
        self.end_lr = end_lr
    
    def on_step_end(self, args, state, control, **kwargs):
        # Log current learning rate
        self.lrs.append(state.learning_rate)
        
        # Get loss from logs
        if len(state.log_history) > 0:
            self.losses.append(state.log_history[-1].get('loss', float('inf')))
        
        # Linearly increase LR
        progress = state.global_step / args.max_steps
        new_lr = self.start_lr * (self.end_lr / self.start_lr) ** progress
        args.learning_rate = new_lr

def plot_lr_finder(lr_finder: LRFinderCallback):
    """Plot learning rate finder results."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Linear scale
    ax1.plot(lr_finder.lrs, lr_finder.losses)
    ax1.set_xlabel("Learning Rate")
    ax1.set_ylabel("Loss")
    ax1.set_title("LR Finder (Linear Scale)")
    ax1.grid(True, alpha=0.3)
    
    # Log scale (more useful)
    ax2.semilogx(lr_finder.lrs, lr_finder.losses)
    ax2.set_xlabel("Learning Rate (log scale)")
    ax2.set_ylabel("Loss")
    ax2.set_title("LR Finder (Log Scale)")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Find optimal LR (where loss decreases fastest)
    losses = np.array(lr_finder.losses)
    lrs = np.array(lr_finder.lrs)
    
    # Calculate slope
    slopes = np.diff(losses) / np.diff(np.log(lrs[1:]))
    best_idx = np.argmin(slopes)  # Most negative slope
    
    print(f"Optimal learning rate: {lrs[best_idx+1]:.2e}")
    return lrs[best_idx+1]

In [ ]:
# Run LR Finder (small model, few steps)
lr_finder_callback = LRFinderCallback(start_lr=1e-8, end_lr=1e-3)

training_args = TrainingArguments(
    output_dir="./lr-finder",
    per_device_train_batch_size=4,
    max_steps=100,  # Run for 100 steps
    logging_steps=10,
    report_to="none",
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)
trainer.add_callback(lr_finder_callback)

print("Running LR Finder...")
trainer.train()

# Plot results
optimal_lr = plot_lr_finder(lr_finder_callback)

## 3. LoRA Configuration

In [ ]:
# Reload model for LoRA training
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                      # Rank of update matrices
    lora_alpha=32,             # Scaling factor (typically 2x rank)
    target_modules=[           # Which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",     # MLP
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Compare parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Gradient Accumulation Demo

In [ ]:
# Compare memory usage with different gradient accumulation settings

def train_with_grad_accum(accum_steps: int):
    """Train with specified gradient accumulation steps."""
    training_args = TrainingArguments(
        output_dir=f"./grad-accum-{accum_steps}",
        per_device_train_batch_size=2,  # Small batch
        gradient_accumulation_steps=accum_steps,
        max_steps=50,
        logging_steps=10,
        report_to="none",
        fp16=False,
    )
    
    # Fresh model for each run
    m = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    m = get_peft_model(m, lora_config)
    
    trainer = Trainer(model=m, args=training_args, train_dataset=tokenized_dataset)
    
    # Measure memory
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    trainer.train()
    
    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"Accum steps={accum_steps}: Peak memory = {peak_mem:.2f} GB")
    
    return trainer.state.log_history

# Demo with different accumulation steps
print("Training with gradient accumulation...")
history_accum_4 = train_with_grad_accum(accum_steps=4)
history_accum_8 = train_with_grad_accum(accum_steps=8)

## 5. Training Monitoring

In [ ]:
def plot_training_metrics(log_history):
    """Plot training loss and learning rate over steps."""
    # Filter logs with loss
    loss_logs = [log for log in log_history if 'loss' in log]
    
    steps = [log['step'] for log in loss_logs]
    losses = [log['loss'] for log in loss_logs]
    lrs = [log.get('learning_rate', 0) for log in loss_logs]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curve
    ax1.plot(steps, losses, 'b-', linewidth=2)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(steps, losses, 3)
    p = np.poly1d(z)
    ax1.plot(steps, p(steps), 'r--', alpha=0.5, label='Trend')
    ax1.legend()
    
    # LR curve
    if any(lrs):
        ax2.plot(steps, lrs, 'g-', linewidth=2)
        ax2.set_xlabel('Step')
        ax2.set_ylabel('Learning Rate')
        ax2.set_title('Learning Rate Schedule')
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot metrics from previous training
plot_training_metrics(history_accum_4)

In [ ]:
# Compute perplexity
def compute_perplexity(model, dataset, tokenizer, num_samples=50):
    """Compute perplexity on a subset of dataset."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            input_ids = torch.tensor([dataset[i]['input_ids']])
            attention_mask = torch.tensor([dataset[i]['attention_mask']])
            labels = torch.tensor([dataset[i]['labels']])
            
            if torch.cuda.is_available():
                input_ids = input_ids.cuda()
                attention_mask = attention_mask.cuda()
                labels = labels.cuda()
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
    
    avg_loss = total_loss / min(num_samples, len(dataset))
    perplexity = torch.exp(torch.tensor(avg_loss))
    
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Perplexity: {perplexity.item():.2f}")
    return perplexity.item()

# Compute perplexity after training
print("Computing perplexity...")
ppl = compute_perplexity(model, tokenized_dataset, tokenizer)

## 6. Catastrophic Forgetting Prevention

In [ ]:
# Simple replay buffer to prevent forgetting
from torch.utils.data import ConcatDataset, DataLoader

# Load some pre-training data for replay
print("Loading replay data...")
replay_dataset = load_dataset("c4", "en", split="train[:100]")

# Tokenize replay data
def tokenize_replay(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

replay_tokenized = replay_dataset.map(tokenize_replay, batched=True, remove_columns=replay_dataset.column_names)
replay_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Add dummy labels for replay
replay_tokenized = replay_tokenized.map(lambda x: {"labels": x["input_ids"].clone()})

print(f"Replay dataset: {len(replay_tokenized)} samples")

# Mix fine-tuning and replay data
mixed_dataset = ConcatDataset([tokenized_dataset, replay_tokenized])
print(f"Mixed dataset: {len(mixed_dataset)} samples")

In [ ]:
# Train with replay to prevent forgetting
training_args = TrainingArguments(
    output_dir="./with-replay",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    logging_steps=10,
    report_to="none",
    fp16=False,
)

# Fresh model with LoRA
model_replay = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
model_replay = get_peft_model(model_replay, lora_config)

trainer_replay = Trainer(
    model=model_replay,
    args=training_args,
    train_dataset=mixed_dataset,
)

print("Training with replay data...")
history_replay = trainer_replay.train().history

print("Training complete!")

## 7. Multi-GPU Setup with Accelerate

In [ ]:
# Multi-GPU training script (save as train_multi_gpu.py)
multi_gpu_script = '''
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

# Initialize accelerator
accelerator = Accelerator(
    gradient_accumulation_steps=4,
    mixed_precision="fp16" if torch.cuda.is_available() else "no",
)

print(f"Number of GPUs: {accelerator.num_processes}")
print(f"Device: {accelerator.device}")

# Load model and data
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("mlabonne/FineTome-100k", split="train[:500]")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

dataloader = DataLoader(tokenized_dataset, batch_size=4, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-4)

# Prepare everything for distributed training
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

# Training loop
num_epochs = 1
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for step, batch in enumerate(tqdm(dataloader, disable=not accelerator.is_local_main_process)):
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss
            
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            
            if step % 10 == 0:
                avg_loss = total_loss / (step + 1)
                accelerator.print(f"Epoch {epoch}, Step {step}, Loss {avg_loss:.4f}")

accelerator.save_model(model, "./multi-gpu-model")
accelerator.print("Training complete and model saved!")
'''

print("Multi-GPU training script:")
print(multi_gpu_script)

# Save the script
with open("train_multi_gpu.py", "w") as f:
    f.write(multi_gpu_script)

print("\nScript saved to train_multi_gpu.py")
print("Run with: accelerate launch train_multi_gpu.py")

## 8. Complete Training Pipeline

In [ ]:
class TrainingPipeline:
    """Complete training pipeline with monitoring."""
    
    def __init__(self, model_name: str, use_lora: bool = True):
        self.model_name = model_name
        self.use_lora = use_lora
        self.model = None
        self.tokenizer = None
        self.training_history = None
    
    def load_model(self):
        """Load model and tokenizer."""
        print(f"Loading model: {self.model_name}")
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        if self.use_lora:
            lora_config = LoraConfig(
                r=16,
                lora_alpha=32,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                lora_dropout=0.05,
                task_type=TaskType.CAUSAL_LM,
            )
            self.model = get_peft_model(self.model, lora_config)
            self.model.print_trainable_parameters()
        
        return self
    
    def load_data(self, dataset_name: str, max_samples: int = 500):
        """Load and tokenize dataset."""
        print(f"Loading dataset: {dataset_name}")
        dataset = load_dataset(dataset_name, split=f"train[:{max_samples}]")
        
        def tokenize(example):
            return self.tokenizer(
                example.get("text", str(example)),
                truncation=True,
                max_length=256,
                padding="max_length"
            )
        
        self.tokenized_dataset = dataset.map(tokenize, batched=True)
        self.tokenized_dataset.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"]
        )
        
        print(f"Loaded {len(self.tokenized_dataset)} samples")
        return self
    
    def train(self, output_dir: str = "./output", **kwargs):
        """Train the model."""
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=kwargs.get("batch_size", 4),
            gradient_accumulation_steps=kwargs.get("grad_accum", 4),
            learning_rate=kwargs.get("lr", 2e-4),
            max_steps=kwargs.get("max_steps", 100),
            logging_steps=kwargs.get("log_steps", 10),
            report_to="none",
            fp16=False,
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.tokenized_dataset,
        )
        
        print("Starting training...")
        trainer.train()
        self.training_history = trainer.state.log_history
        
        trainer.save_model(output_dir)
        print(f"Model saved to {output_dir}")
        
        return self
    
    def evaluate(self, test_samples: int = 50):
        """Evaluate model perplexity."""
        return compute_perplexity(
            self.model,
            self.tokenized_dataset,
            self.tokenizer,
            num_samples=test_samples
        )
    
    def plot_metrics(self):
        """Plot training metrics."""
        if self.training_history:
            plot_training_metrics(self.training_history)
        else:
            print("No training history available")

# Example usage
pipeline = TrainingPipeline(model_name, use_lora=True)
pipeline.load_model()
pipeline.load_data("mlabonne/FineTome-100k", max_samples=200)
pipeline.train(output_dir="./pipeline-output", batch_size=2, max_steps=50)
pipeline.evaluate()
pipeline.plot_metrics()

## Summary

This notebook covered:

1. **SFT Workflow**: Loading model, tokenizer, dataset, and training
2. **Learning Rate Finder**: Automated LR search with visualization
3. **LoRA Configuration**: Parameter-efficient fine-tuning setup
4. **Gradient Accumulation**: Memory optimization technique
5. **Training Monitoring**: Loss curves, perplexity, LR schedules
6. **Catastrophic Forgetting**: Replay buffer prevention method
7. **Multi-GPU Training**: Accelerate-based distributed training
8. **Complete Pipeline**: End-to-end training class

### Key Takeaways:

- Use LR finder to identify optimal learning rate before full training
- LoRA reduces trainable parameters by 95%+ with minimal performance loss
- Gradient accumulation trades time for memory efficiency
- Replay data helps prevent catastrophic forgetting
- Accelerate simplifies multi-GPU training